In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from numba import njit
from numba_progress import ProgressBar

from filters import (
    # parameter dtypes
    NLMS_params,
    sKF_params,
    sKF_L_params,
    skf_int_params,
    skf_L_int_params,
    # signal / environment helpers
    autocorr_matrix_calc,
    autocorr_matrix_estimate,
    AR_settling_time,
    std_behavior,
    # algorithms
    NLMS_algorithm,
    sKF_algorithm,
    sKF_L_algorithm,
    sKF_L_exact_algorithm,
    sKF_integral_algorithm,
    sKF_L_integral_algorithm,
    # monte carlo driver
    MC_Simulations_Modular_Variance,
)

In [ ]:

N=200
L=3
ho = np.sinc(np.linspace(0, 1, L))
ho = ho / np.linalg.norm(ho)
h0 = np.zeros(L)
# var_x = 1
# var_v = 1e-3
AR = np.array([1.0, -0.9, 0.95, -0.8, 0.8])

##### GRID SEARCH

In [ ]:
import itertools, pickle, os

GRID = {
    "NR":                 [1],
    "epsilon":            [0.01],
    "var_theta_0":        [2.0],
    "var_eta":            [1e-3],
    "dx_factor":          [0.1],
    "min_std_deviations": [3],
    "var_x":              [1],
    "var_v":              [1e-3],
    # "b_eta":               [2],
}

# CACHE = "grid_results.pkl"
# results = pickle.load(open(CACHE, "rb")) if os.path.exists(CACHE) else {}
results = {}

@njit
def numba_seed(s): np.random.seed(s)

N=200

for combo in itertools.product(*GRID.values()):
    # if combo in results:
    #     continue
    NR_, eps, vt0, veta, dxf, msd, var_x, var_v = combo

    p_int   = np.void(("sKF_integral", eps, vt0, veta, dxf, msd), dtype=skf_int_params)
    p_paper = np.void(("sKF_paper",    eps, veta, vt0),           dtype=sKF_params)

    numba_seed(0)
    results[combo] = MC_Simulations_Modular_Variance(
        N, NR_, ho, var_x, var_v, h0,
        [sKF_integral_algorithm, sKF_algorithm],
        [p_int, p_paper], AR)

    #pickle.dump(results, open(CACHE, "wb"))   # save each cell, crash-safe
    print(combo, "done")

In [ ]:
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

def grid_plot(draw, title, sharey=True, ncol=2):
    combos = list(results.keys())
    nrow = int(np.ceil(len(combos)/ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(7*ncol, 4*nrow),
                             sharey=sharey, squeeze=False,
                             constrained_layout=True)
    for ax, combo in zip(axes.ravel(), combos):
        draw(ax, results[combo])
        items = [f"{k}={v:g}" for k, v in zip(GRID, combo)]
        ax.set_title(", ".join(items[:3]) + "\n" + ", ".join(items[3:]), fontsize=9)
        ax.grid(alpha=0.3)
    for ax in axes.ravel()[len(combos):]:
        ax.axis("off")
    axes[0, 0].legend(fontsize=7)
    fig.suptitle(title)
    plt.show()


def draw_weights(ax, r):                                    # cell 32
    num, pap = r["sKF_integral"]["h"], r["sKF_paper"]["h"]
    for k in range(num.shape[1]):
        c = colors[k % len(colors)]
        ax.plot(num[:, k], color=c, ls='-',  label=f"$w_{k}$ num")
        ax.plot(pap[:, k], color=c, ls='--', label=f"$w_{k}$ paper")
        ax.axhline(ho[k], color=c, ls=':', lw=1)

def draw_weights_diff(ax, r):                               # cell 33
    num, pap = r["sKF_integral"]["h"], r["sKF_paper"]["h"]
    for k in range(num.shape[1]):
        # Compute difference:
        diff = np.abs(num[:, k] - pap[:, k])
        ax.plot(10*np.log10(diff / np.abs(pap[:, k])),
                color=colors[k % len(colors)], label=f"$w_{k}$")
        print(f"mean diff w_{k}: {np.mean(diff):.4g}")
    ax.axhline(10*np.log10(0.012), ls=":", c="k")           # regressor floor, 1.2%

def draw_var(ax, r):                                        # cell 34
    ax.plot(r["sKF_integral"]["var"][:, 0], '-',  label="v num")
    ax.plot(r["sKF_paper"]["var"][:, 0],    '--', label="v paper")

def draw_var_diff(ax, r):                                   # cell 36
    vn, vp = r["sKF_integral"]["var"][:, 0], r["sKF_paper"]["var"][:, 0]
    ax.plot(10*np.log10(np.abs(vn - vp) / np.abs(vp)))
    ax.axhline(10*np.log10(0.012), ls=":", c="k")


grid_plot(draw_weights,      "sKF weights: numerical integration vs paper recursion")
grid_plot(draw_weights_diff, "sKF weights difference (num-paper)/paper [dB]")
grid_plot(draw_var,          "sKF variance: numerical integration vs paper recursion",
          sharey=False)
grid_plot(draw_var_diff,     "sKF variance difference (num-paper)/paper [dB]")

In [ ]:
# GUARDAR EN DISCO !!!

DRAWS = [(draw_weights,      "weights: num vs paper"),
         (draw_weights_diff, "weights diff [dB]"),
         (draw_var,          "variance: num vs paper"),
         (draw_var_diff,     "variance diff [dB]")]

def grid_svg(fname="grid.svg", combos=None):
    combos = list(results.keys()) if combos is None else combos
    nrow, ncol = len(combos), len(DRAWS)
    fig, axes = plt.subplots(nrow, ncol, figsize=(7*ncol, 4*nrow),
                             sharey='col', squeeze=False,
                             constrained_layout=True)
    for i, combo in enumerate(combos):
        items = [f"{k}={v:g}" for k, v in zip(GRID, combo)]
        label = ", ".join(items[:3]) + "\n" + ", ".join(items[3:])
        for j, (draw, name) in enumerate(DRAWS):
            draw(axes[i, j], results[combo])
            axes[i, j].set_title(f"{name}\n{label}", fontsize=8)
            axes[i, j].grid(alpha=0.3)
    axes[0, 0].legend(fontsize=7)
    fig.savefig(fname, format="svg")
    plt.close(fig)

grid_svg("grid.svg")